# 시군구 응급의료 접근성 × 복지 취약 클러스터링

- 입력: `data/processed/sigungu_table.csv` (`python -m src.build_table` 로 생성)
- 출력: `outputs/` 아래 k 선정 그래프, 클러스터 평균 표, 점수표, 이중 취약지역 목록과 산점도

In [ ]:
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# 한글 폰트: OS 기본 폰트 사용
plt.rc('font', family={'Windows': 'Malgun Gothic', 'Darwin': 'AppleGothic'}.get(platform.system(), 'NanumGothic'))
plt.rcParams['axes.unicode_minus'] = False

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'sigungu_table.csv'
OUT = ROOT / 'outputs'
SEED = 195   # legacy 분석과 동일
TOP = 0.25   # 이중 취약 기준: 두 점수 모두 상위 25%

## 1. 피처: 인구 대비 지표

| 피처 | 계산 | 취약 방향 |
|---|---|---|
| 10만명당_응급의료기관 | 응급의료기관 수 / 총인구 × 10만 | 낮을수록 |
| 평균접근거리_km | 국토모니터링 응급의료시설 접근성 | 높을수록 |
| 고령인구비율 | 65세 이상 / 총인구 × 100 | 높을수록 |
| 독거노인비율 | 독거노인 / **65세 이상** × 100 | 높을수록 |
| 장애인비율 | 등록장애인 / 총인구 × 100 | 높을수록 |
| 수급자비율 | 기초생활수급자 / 총인구 × 100 | 높을수록 |

In [ ]:
t = pd.read_csv(DATA, dtype={'code': str}).set_index('code')
df = pd.DataFrame({
    '지역': (t['시도'] + ' ' + t['시군구'].fillna('')).str.strip(),
    '10만명당_응급의료기관': t['응급의료기관수'] / t['총인구'] * 1e5,
    '평균접근거리_km': t['응급의료시설_평균접근거리_km'],
    '고령인구비율': t['65세이상인구'] / t['총인구'] * 100,
    '독거노인비율': t['독거노인수'] / t['65세이상인구'] * 100,
    '장애인비율': t['등록장애인수'] / t['총인구'] * 100,
    '수급자비율': t['기초수급자수'] / t['총인구'] * 100,
})
FEATURES = df.columns[1:].tolist()
ACCESS = ['10만명당_응급의료기관', '평균접근거리_km']
WELFARE = ['고령인구비율', '독거노인비율', '장애인비율', '수급자비율']

missing = df[df[FEATURES].isna().any(axis=1)]
print(f'결측으로 제외: {len(missing)}개', missing['지역'].tolist())
df = df.dropna(subset=FEATURES)
print(f'분석 대상: {len(df)}개 시군구')
df[FEATURES].describe().round(2)

## 2. k 선정: 엘보우 + 실루엣

표준화 후 k=2~10을 비교한다. 기본값은 실루엣 최대 k이고, 엘보우 그래프를 보고 해석이 더 나은 k가 있으면 `K` 를 직접 바꾼다.

In [ ]:
OUT.mkdir(exist_ok=True)
X = StandardScaler().fit_transform(df[FEATURES])
ks = list(range(2, 11))
models = {k: KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X) for k in ks}
inertia = [models[k].inertia_ for k in ks]
sil = [silhouette_score(X, models[k].labels_) for k in ks]
K = ks[int(np.argmax(sil))]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, y, title in [(axes[0], inertia, '엘보우 (군집 내 제곱합)'), (axes[1], sil, '실루엣 점수')]:
    ax.plot(ks, y, color='#2a78d6', lw=2, marker='o', ms=8)
    ax.axvline(K, color='#52514e', ls='--', lw=1)
    ax.set(title=title, xlabel='k', xticks=ks)
    ax.grid(axis='y', color='#e5e4df', lw=0.8)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle(f'k 선정 (선택 k={K})')
fig.tight_layout()
fig.savefig(OUT / 'k_selection.png', dpi=200)
print(pd.DataFrame({'k': ks, 'inertia': inertia, 'silhouette': sil}).round(3).to_string(index=False))

In [ ]:
km = models[K]
df['클러스터'] = km.labels_
profile = df.groupby('클러스터')[FEATURES].mean().round(2)
profile.insert(0, '시군구수', df['클러스터'].value_counts().sort_index())
profile.to_csv(OUT / 'cluster_profile.csv', encoding='utf-8-sig')
profile

## 3. 이중 취약지역

- **접근성 취약 점수** = ( z(평균접근거리) − z(10만명당 응급의료기관) ) / 2
- **복지 취약 점수** = z(고령인구비율, 독거노인비율, 장애인비율, 수급자비율)의 평균
- z는 위 표준화 값이고, 두 점수 모두 높을수록 취약하다.
- **이중 취약지역**은 두 점수가 모두 상위 `TOP`(기본 25%) 안에 드는 시군구다.

In [ ]:
z = pd.DataFrame(X, index=df.index, columns=FEATURES)
df['접근성취약점수'] = (z['평균접근거리_km'] - z['10만명당_응급의료기관']) / 2
df['복지취약점수'] = z[WELFARE].mean(axis=1)
a_cut = df['접근성취약점수'].quantile(1 - TOP)
w_cut = df['복지취약점수'].quantile(1 - TOP)
df['이중취약'] = (df['접근성취약점수'] >= a_cut) & (df['복지취약점수'] >= w_cut)

df.round(3).to_csv(OUT / 'sigungu_scores.csv', encoding='utf-8-sig')
dv = df[df['이중취약']].assign(합산=lambda d: d['접근성취약점수'] + d['복지취약점수']).sort_values('합산', ascending=False)
dv.round(3).to_csv(OUT / 'double_vulnerable.csv', encoding='utf-8-sig')
print(f'이중 취약지역 {len(dv)}개 (기준: 접근성 ≥ {a_cut:.2f}, 복지 ≥ {w_cut:.2f})')
dv[['지역', '접근성취약점수', '복지취약점수', '클러스터'] + FEATURES].round(2)

In [ ]:
LABEL_TOP = 10  # 이름을 붙일 이중 취약지역 수 (합산 점수 순)

fig, ax = plt.subplots(figsize=(9, 7))
rest = df[~df['이중취약']]
ax.scatter(rest['복지취약점수'], rest['접근성취약점수'], s=36, color='#b5b4ad', edgecolor='white', lw=0.8, label='그 외 시군구')
ax.scatter(dv['복지취약점수'], dv['접근성취약점수'], s=56, color='#eb6834', edgecolor='white', lw=0.8, label=f'이중 취약지역 ({len(dv)}개)')
ax.axvline(w_cut, color='#52514e', ls='--', lw=1)
ax.axhline(a_cut, color='#52514e', ls='--', lw=1)
ax.set(xlabel='복지 취약 점수 →  취약', ylabel='응급의료 접근성 취약 점수 →  취약',
       title=f'시군구별 응급의료 접근성 × 복지 취약 (점선: 상위 {TOP:.0%} 기준)')
ax.grid(color='#e5e4df', lw=0.8)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(loc='lower right', frameon=False)
fig.tight_layout()

# 이름표: 다른 이름표와 겹치지 않는 위치를 차례로 시도, 다 겹치면 생략 (생략된 지역은 double_vulnerable.csv에 있음)
renderer, placed = fig.canvas.get_renderer(), []
for _, r in dv.head(LABEL_TOP).iterrows():
    for dx, dy, ha in [(5, 4, 'left'), (5, -11, 'left'), (-5, 4, 'right'), (-5, -11, 'right')]:
        txt = ax.annotate(r['지역'], (r['복지취약점수'], r['접근성취약점수']), xytext=(dx, dy),
                          textcoords='offset points', ha=ha, fontsize=8, color='#0b0b0b')
        box = txt.get_window_extent(renderer)
        if not any(box.overlaps(b) for b in placed):
            placed.append(box)
            break
        txt.remove()
fig.savefig(OUT / 'double_vulnerable_scatter.png', dpi=200)